## Setup

In [192]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Define path to data/outputs folder
DATA_DIR = Path.cwd().parents[2] / 'data' / 'spreadsheets'
OUTPUT_DIR = Path.cwd().parents[2] / 'estimation' / 'quick_estimation' / 'outputs'
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

# Read source files
city_pop22 = pd.read_excel(DATA_DIR / 'NGA_citypop_pop2022.xlsx', sheet_name='Sheet1')      # CityPopulation.de 2022 Population Projection Data
cod        = pd.read_excel(DATA_DIR / 'NGA_PGR_2022.xlsx',        sheet_name='PGR_Proj')    # COD-PS 2022 Population Growth Rate Projections
city       = pd.read_excel(DATA_DIR / 'NGA_citypop_apc.xlsx',     sheet_name='cp_apc')      # CityPopulation.de Annual Population Change Data
pol_map    = pd.read_excel(DATA_DIR / 'nga_lga_pol_map.xlsx')                               # LGA to Political Map

cod = cod.rename(columns={'PGR_BOTH_2022': 'ANN_POP_CHANGE'})

DATA_DIR: /Users/jarretangbazo/nga_population_estimation/data/spreadsheets
OUTPUT_DIR: /Users/jarretangbazo/nga_population_estimation/estimation/quick_estimation/outputs


## 1. Compare Growth Rate Sources

In [193]:
pop_growth_merged = city.merge(cod, on='ADM1_NAME', how='outer', suffixes=('_city', '_cod'))

missing_in_cod  = pop_growth_merged[pop_growth_merged['ANN_POP_CHANGE_cod'].isna()][['ADM1_NAME','ANN_POP_CHANGE_city']]
missing_in_city = pop_growth_merged[pop_growth_merged['ANN_POP_CHANGE_city'].isna()][['ADM1_NAME','ANN_POP_CHANGE_cod']]
print("Missing from COD-PS:\n", missing_in_cod)
print("\nMissing from City Population:\n", missing_in_city)

Missing from COD-PS:
    ADM1_NAME  ANN_POP_CHANGE_city
25  Nasarawa                  2.8

Missing from City Population:
 Empty DataFrame
Columns: [ADM1_NAME, ANN_POP_CHANGE_cod]
Index: []


In [194]:
both = pop_growth_merged.dropna(subset=['ANN_POP_CHANGE_city','ANN_POP_CHANGE_cod']).copy()
both['abs_diff'] = (both['ANN_POP_CHANGE_city'] - both['ANN_POP_CHANGE_cod']).abs()
both['within_rounding'] = both['abs_diff'] <= 0.1

print("Max absolute difference: ", both['abs_diff'].max())
print("Mean absolute difference:", both['abs_diff'].mean())
print("Within rounding:         ", both['within_rounding'].sum(), "of", len(both))
print("Correlation:             ", both['ANN_POP_CHANGE_city'].corr(both['ANN_POP_CHANGE_cod']).round(4))

outliers = both[~both['within_rounding']]
print("\nOutliers:\n", outliers[['ADM1_NAME','ANN_POP_CHANGE_city','ANN_POP_CHANGE_cod','abs_diff']])

Max absolute difference:  5.2
Mean absolute difference: 0.19194444444444442
Within rounding:          33 of 36
Correlation:              0.7154

Outliers:
                     ADM1_NAME  ANN_POP_CHANGE_city  ANN_POP_CHANGE_cod  \
14  Federal Capital Territory                  5.0                4.88   
17                     Jigawa                  3.5                3.40   
36                    Zamfara                  3.7                8.90   

    abs_diff  
14      0.12  
17      0.10  
36      5.20  


## 2. Reconcile Growth Rates

Nasarawa is missing from COD-PS data. Fill annual population growth rate from City Population data.
  
Zamfara has conflicting rates (City Pop: 3.7%, COD-PS: 8.9%). We will use the City Population data (lower estimate for annual population growth rate).

In [195]:
# Fill Nasarawa from City Population
pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Nasarawa', 'ANN_POP_CHANGE_cod'] = \
    pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Nasarawa', 'ANN_POP_CHANGE_city']

# Zamfara: trust City Population rate (3.7%)
# City Population rate = 3.7, COD-PS rate = 8.9 — using City Population
pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Zamfara', 'rate_final'] = 3.7

# All other states: use COD-PS rate
pop_growth_merged['rate_final'] = pop_growth_merged['rate_final'].fillna(pop_growth_merged['ANN_POP_CHANGE_cod'])

print(pop_growth_merged[['ADM1_NAME','rate_final']].to_string())
print("\nNull rates:", pop_growth_merged['rate_final'].isna().sum())

                    ADM1_NAME  rate_final
0                        Abia        2.35
1                     Adamawa        2.71
2                   Akwa Ibom        1.52
3                     Anambra        2.21
4                      Bauchi        3.62
5                     Bayelsa        2.49
6                       Benue        2.30
7                       Borno        2.39
8                 Cross River        2.63
9                       Delta        1.97
10                     Ebonyi        2.49
11                        Edo        2.44
12                      Ekiti        2.53
13                      Enugu        2.26
14  Federal Capital Territory        4.88
15                      Gombe        3.22
16                        Imo        2.06
17                     Jigawa        3.40
18                     Kaduna        2.44
19                       Kano        3.11
20                    Katsina        3.63
21                      Kebbi        3.35
22                       Kogi     

## 3. LGA Population Projections (2022 → 2025)

In [196]:
# Standardize state names to uppercase before merge
pop2022_clean = city_pop22.rename(columns={'STATE': 'ADM1_NAME', 'POP_2022': 'POP22'}).copy()
pop2022_clean['ADM1_NAME'] = pop2022_clean['ADM1_NAME'].str.upper()
pop_growth_merged['ADM1_NAME'] = pop_growth_merged['ADM1_NAME'].str.upper()

# Verify state name alignment
states_growth = set(pop_growth_merged['ADM1_NAME'])
states_pop    = set(pop2022_clean['ADM1_NAME'])
print("In growth not in pop2022:", states_growth - states_pop)
print("In pop2022 not in growth:", states_pop - states_growth)

In growth not in pop2022: set()
In pop2022 not in growth: set()


In [197]:
merged = pop2022_clean.merge(pop_growth_merged[['ADM1_NAME','rate_final']], on='ADM1_NAME', how='left')

merged_final = merged[['ADM1_NAME','LGA','POP22','rate_final']].rename(
    columns={'rate_final': 'ANNUAL_GROWTH_RATE22'}).copy()

# Project 2022 population forward 3 years to end of 2025
merged_final['POP25_PROJECTED'] = merged_final['POP22'] * (1 + merged_final['ANNUAL_GROWTH_RATE22'] / 100) ** 3

# KEY FIX: uppercase LGA for matching with pol_map later
merged_final['LGA_UPPER'] = merged_final['LGA'].str.upper()

print(f"LGAs with projection: {merged_final['POP25_PROJECTED'].notna().sum()} of {len(merged_final)}")
print(f"Nigeria total POP25:  {merged_final['POP25_PROJECTED'].sum():,.0f}")
print(merged_final.head())

LGAs with projection: 775 of 775
Nigeria total POP25:  235,207,341
  ADM1_NAME        LGA   POP22  ANNUAL_GROWTH_RATE22  POP25_PROJECTED  \
0      ABIA  Aba North  155600                  2.35    166829.609657   
1      ABIA  Aba South  622400                  2.35    667318.438629   
2      ABIA  Arochukwu  246600                  2.35    264397.054894   
3      ABIA      Bende  280500                  2.35    300743.608669   
4      ABIA    Ikwuano  200800                  2.35    215291.681357   

   LGA_UPPER  
0  ABA NORTH  
1  ABA SOUTH  
2  AROCHUKWU  
3      BENDE  
4    IKWUANO  


## 4. Political Map Cleaning

Transforms the raw Wikipedia constituency crosswalk into a clean STATE / LGA / SEN_DIST / FED_CONST table.

Steps:
1. **State extraction** — strips trailing direction words iteratively (e.g., strips `"Akwa Ibom North East"` to `Akwa Ibom North` to `"Akwa Ibom"`)
2. **Direction expansion** — `"Awka North/South"` to `["Awka North"`, `"Awka South"]`  
3. **One-to-many tokens** — four tokens that cover two LGAs each (e.g., `"Birniwa Guri"` to Biriniwa + Guri)
4. **Name crosswalk** — 100+ naming differences between Wikipedia and citypopulation.de
5. **Dedup + split counting** — avoids double-counting combined LGAs

In [198]:
# Step 1: State extraction
def extract_state(sd):
    """Strip trailing direction words iteratively: 'Akwa Ibom North East' -> 'Akwa Ibom'"""
    if pd.isna(sd):
        return None
    parts = str(sd).split()
    dir_words = {'North', 'South', 'East', 'West', 'Central'}
    while parts and parts[-1] in dir_words:
        parts.pop()
    name = ' '.join(parts)
    return 'Federal Capital Territory' if name == 'FCT' else name

pol_map['State'] = pol_map['Senatorial District'].apply(extract_state)

In [199]:
# Step 2: Expand abbreviated direction suffixes
# e.g. "Awka North/South" -> ["Awka North", "Awka South"]
# e.g. "Nnewi North/South/Ekwusigo" -> ["Nnewi North", "Nnewi South", "Ekwusigo"]
directions  = ['North', 'South', 'East', 'West', 'Central']
dir_set     = set(directions)
dir_pattern = r'\s+(' + '|'.join(directions) + r')$'

def expand_constituency_lgas(c):
    if pd.isna(c):
        return []
    parts = [p.strip() for p in str(c).split('/')]
    expanded = []
    for part in parts:
        if part in dir_set and expanded:
            base = re.sub(dir_pattern, '', expanded[-1]).strip()
            part = base + ' ' + part
        expanded.append(part)
    return expanded

pol_map['LGA'] = pol_map['Constituency'].apply(expand_constituency_lgas)
pol_map_long   = pol_map.explode('LGA')
pol_map_long['LGA'] = pol_map_long['LGA'].str.strip()

pol_map_final = pol_map_long.rename(columns={
    'State':               'STATE',
    'Senatorial District': 'SEN_DIST',
    'Constituency':        'FED_CONST'
})[['STATE', 'LGA', 'SEN_DIST', 'FED_CONST']].drop_duplicates().reset_index(drop=True)

# IS_SPLIT flag for LGAs ending in I / II (e.g. Mushin I, Mushin II)
pol_map_final['IS_SPLIT'] = pol_map_final['LGA'].str.contains(r'\s+I{1,2}$', regex=True)
pol_map_final['LGA_MERGE'] = pol_map_final['LGA'].str.replace(r'\s+I{1,2}$', '', regex=True).str.strip()

# Uppercase for matching
pol_map_final['STATE']     = pol_map_final['STATE'].str.upper()
pol_map_final['LGA_MERGE'] = pol_map_final['LGA_MERGE'].str.upper()

# Normalise curly apostrophe → straight (Qua'an Pan encoding difference)
pol_map_final['LGA_MERGE'] = pol_map_final['LGA_MERGE'].str.replace('\u2019', "'", regex=False)

In [200]:
# Step 3: Expand one-to-many tokens
# Four constituency names cover 2 pop2022 LGAs each but weren't separated by "/"
ONE_TO_MANY = {
    'KWAYA-KUSAR SHANI': ['KWAYA KUSAR', 'SHANI'],
    'IVO-OHAOZARA':      ['IVO',         'OHAOZARA'],
    'KWALI GWAGWALADA':  ['KWALI',       'GWAGWALADA'],
    'BIRNIWA GURI':      ['BIRINIWA',    'GURI'],
}

expanded_rows = []
for _, row in pol_map_final.iterrows():
    if row['LGA_MERGE'] in ONE_TO_MANY:
        for lga in ONE_TO_MANY[row['LGA_MERGE']]:
            new_row = row.copy()
            new_row['LGA_MERGE'] = lga
            expanded_rows.append(new_row)
    else:
        expanded_rows.append(row)

pol_map_final = pd.DataFrame(expanded_rows).reset_index(drop=True)

In [201]:
# Step 4: Name crosswalk
# Resolves all naming differences between Wikipedia pol_map and citypopulation.de
NAME_CROSSWALK = {
    ('ABIA',                      'ISIALA NGWA NORTH'):      'ISIALA-NGWA NORTH',
    ('ABIA',                      'ISIALA NGWA SOUTH'):      'ISIALA-NGWA SOUTH',
    ('ABIA',                      'OBINGWA'):                'OBI NGWA',
    ('ABIA',                      'OSISIOMA'):               'OSISIOMA NGWA',
    ('ABIA',                      'UMUNNEOCHI'):             'UMU-NNEOCHI',
    ('ADAMAWA',                   'MAYO BELWA'):             'MAYO-BELWA',
    ('AKWA IBOM',                 'URUE OFFONG'):            'URUE-OFFONG/ORUKO',
    ('AKWA IBOM',                 'ORUKO'):                  'URUE-OFFONG/ORUKO',
    ('BAUCHI',                    'DAMBAM'):                 'DAMBAN',
    ('BAUCHI',                    'GUNJUWA'):                'GANJUWA',
    ('BAUCHI',                    'ITAS-GADAU'):             'ITAS/GADAU',
    ('BAUCHI',                    'TAFAWA BALEWA'):          'TAFAWA-BALEWA',
    ('BAYELSA',                   'KOLOKUNA'):               'KOLOKUMA/OPOKUMA',
    ('BAYELSA',                   'OPOKUMA'):                'KOLOKUMA/OPOKUMA',
    ('BENUE',                     'OBADIGBO'):               'OGBADIBO',
    ('BENUE',                     'OTUKPO'):                 'OTURKPO',
    ('BORNO',                     'ASKIRA-UBA'):             'ASKIRA/UBA',
    ('BORNO',                     'GUZAMALI'):               'GUZAMALA',
    ('BORNO',                     'KALA-BALGE'):             'KALA/BALGE',
    ('BORNO',                     'MAIDUGURI METROPOLITAN'): 'MAIDUGURI',
    ('CROSS RIVER',               'BAKASSI'):                'BAKASSI [→ CAMEROON (2008)]',
    ('CROSS RIVER',               'BEKWARRA'):               'BEKWARA',
    ('DELTA',                     'NKOKWA EAST'):            'NDOKWA EAST',
    ('EDO',                       'ESAN SOUTH'):             'ESAN SOUTH EAST',
    ('EKITI',                     'GBONYIN'):                'AIYEKIRE (GBONYIN)',
    ('EKITI',                     'IDO'):                    'IDO-OSI',
    ('EKITI',                     'IFELODUN'):               'IREPODUN/IFELODUN',
    ('EKITI',                     'ILEJEME'):                'ILEJEMEJE',
    ('EKITI',                     'IREPODUN'):               'IREPODUN/IFELODUN',
    ('EKITI',                     'ISE'):                    'ISE/ORUN',
    ('EKITI',                     'ORUN'):                   'ISE/ORUN',
    ('EKITI',                     'OSI MOBA'):               'MOBA',
    ('ENUGU',                     'ISI UZO'):                'ISI-UZO',
    ('ENUGU',                     'OJI RIVER'):              'OJI-RIVER',
    ('FEDERAL CAPITAL TERRITORY', 'ABUJA MUNICIPAL'):        'ABUJA MUNICIPAL AREA COUNCIL',
    ('FEDERAL CAPITAL TERRITORY', 'BWAR'):                   'BWARI',
    ('GOMBE',                     'YAMALTU'):                'YAMALTU/DEBA',
    ('GOMBE',                     'DEBA'):                   'YAMALTU/DEBA',
    ('GOMBE',                     'SHONGOM'):                'SHOMGOM',
    ('IMO',                       'ABOH MBAISE'):            'ABO-MBAISE',
    ('IMO',                       'AHIAZU MBAISE'):          'AHIAZU-MBAISE',
    ('IMO',                       'EHIME MBANO'):            'EHIME-MBANO',
    ('IMO',                       'IHITE-UBOMA'):            'IHITTE/UBOMA',
    ('IMO',                       'ISIALA MBANO'):           'ISIALA-MBANO',
    ('IMO',                       'NGOR OKPALA'):            'NGOR-OKPALA',
    ('IMO',                       'OHAJI-EGBEMA'):           'OHAJI/EGBEMA',
    ('IMO',                       'ONUIMO'):                 'UNUIMO',
    ('JIGAWA',                    'KIRIKASAMMA'):            'KIRI KASAMA',
    ('JIGAWA',                    'MALLAM MADORI'):          'MALAM MADORI',
    ('KADUNA',                    'BIRNIN GWARI'):           'BIRNIN-GWARI',
    ('KADUNA',                    'JEMAA'):                  "JEMA'A",
    ('KADUNA',                    'SABON GARI'):             'SABON-GARI',
    ('KADUNA',                    'ZANGON KATAF'):           'ZANGON-KATAF',
    ('KANO',                      'ALABASU'):                'ALBASU',
    ('KANO',                      'GARUN MALLAM'):           'GARUM MALLAM',
    ('KANO',                      'IKABO'):                  'KABO',
    ('KANO',                      'NASSARAWA'):              'NASARAWA',
    ('KATSINA',                   'DANMUSA'):                'DAN MUSA',
    ('KATSINA',                   'MAIADUA'):                "MAI'ADUA",
    ('KEBBI',                     'AREWA'):                  'AREWA-DANDI',
    ('KEBBI',                     'DANKO'):                  'WASAGU/DANKO',
    ('KEBBI',                     'KOKO-BESSE'):             'KOKO/BESSE',
    ('KEBBI',                     'WASAGU'):                 'WASAGU/DANKO',
    ('KOGI',                      'IGALAMELA ODOLU'):        'IGALAMELA-ODOLU',
    ('KOGI',                      'KABBA-BUNU'):             'KABBA/BUNU',
    ('KOGI',                      'KOTON KARFE'):            'LOKOJA',
    ('KOGI',                      'MOPAMURO'):               'MOPA-MURO',
    ('KOGI',                      'OGORI-MAGOGO'):           'OGORI/MAGONGO',
    ('LAGOS',                     'AJEROMI'):                'AJEROMI-IFELODUN',
    ('LAGOS',                     'IBEJU LEKKI'):            'IBEJU/LEKKI',
    ('LAGOS',                     'IFAKO-IJAIYE'):           'IFAKO-IJAYE',
    ('LAGOS',                     'IFELODUN'):               'AJEROMI-IFELODUN',
    ('LAGOS',                     'ISOLO'):                  'OSHODI-ISOLO',
    ('LAGOS',                     'OSHODI'):                 'OSHODI-ISOLO',
    ('NASARAWA',                  'NASSARAWA'):              'NASARAWA',
    ('NASARAWA',                  'NASSARAWA EGGON'):        'NASARAWA-EGGON',
    ('NIGER',                     'BOOSO'):                  'BOSSO',
    ('NIGER',                     'MUNYA'):                  'MUYA',
    ('NIGER',                     'TAPA'):                   'TAFA',
    ('OGUN',                      'ADO-ODO'):                'ADO-ODO/OTA',
    ('OGUN',                      'EGBADO NORTH'):           'YEWA NORTH (EGBADO NORTH)',
    ('OGUN',                      'EGBADO SOUTH'):           'YEWA SOUTH (EGBADO SOUTH)',
    ('OGUN',                      'IMEKO-AFON'):             'IMEKO AFON',
    ('OGUN',                      'OTA'):                    'ADO-ODO/OTA',
    ('OGUN',                      'SHAGAMU'):                'SAGAMU',
    ('ONDO',                      'ESEODO'):                 'ESE-ODO',
    ('ONDO',                      'ILEOLUJI'):               'ILE-OLUJI-OKEIGBO',
    ('ONDO',                      'OKEIGBO'):                'ILE-OLUJI-OKEIGBO',
    ('OSUN',                      'AYEDAADE'):               'AIYEDAADE',
    ('OSUN',                      'AYEDIRE'):                'AIYEDIRE',
    ('OYO',                       'OGBOMOSO NORTH'):         'OGBOMOSHO NORTH',
    ('OYO',                       'OGBOMOSO SOUTH'):         'OGBOMOSHO SOUTH',
    ('OYO',                       'OGO-OLUWA'):              'OGO OLUWA',
    ('OYO',                       'OLURUNSOGO'):             'OLORUNSOGO',
    ('OYO',                       'ORIRE'):                  'ORI IRE',
    ('PLATEAU',                   'PAN'):                    "QUA'AN PAN",
    ('PLATEAU',                   "QUA'AN"):                 "QUA'AN PAN",
    ('PLATEAU',                   'SHEDAM'):                 'SHENDAM',
    ('RIVERS',                    'ABUA'):                   'ABUA - ODUAL',
    ('RIVERS',                    'AKPOR'):                  'OBIO/AKPOR',
    ('RIVERS',                    'ASARI TORU'):             'ASARI-TORU',
    ('RIVERS',                    'BOLO'):                   'OGU - BOLO',
    ('RIVERS',                    'EGBEMA'):                 'OGBA - EGBEMA - NDONI',
    ('RIVERS',                    'EMOHUA'):                 'EMUOHA',
    ('RIVERS',                    'NDONI'):                  'OGBA - EGBEMA - NDONI',
    ('RIVERS',                    'NKORO'):                  'OPOBO - NKORO',
    ('RIVERS',                    'OBIO'):                   'OBIO/AKPOR',
    ('RIVERS',                    'ODUA'):                   'ABUA - ODUAL',
    ('RIVERS',                    'OGBA'):                   'OGBA - EGBEMA - NDONI',
    ('RIVERS',                    'OGU'):                    'OGU - BOLO',
    ('RIVERS',                    'OMUMA'):                  'OMUMMA',
    ('RIVERS',                    'OPOBO'):                  'OPOBO - NKORO',
    ('RIVERS',                    'PORT HARCOURT'):          'PORT-HARCOURT',
    ('SOKOTO',                    'DANGE-SHUNI'):            'DANGE SHUNI',
    ('TARABA',                    'KARIM LAMIDO'):           'KARIM-LAMIDO',
    ('TARABA',                    'TAKUMA'):                 'TAKUM',
    ('YOBE',                      'TARMUWA'):                'TARMUA',
    ('ZAMFARA',                   'GUNMI'):                  'GUMMI',
    ('ZAMFARA',                   'MAFARA'):                 'TALATA MAFARA',
    ('ZAMFARA',                   'TALATA'):                 'TALATA MAFARA',
}

pol_map_final['LGA_MERGE'] = pol_map_final.apply(
    lambda row: NAME_CROSSWALK.get((row['STATE'], row['LGA_MERGE']), row['LGA_MERGE']),
    axis=1
)

In [202]:
# Step 5: Dedup + constituency count
# Dedup at (FED_CONST, LGA_MERGE): prevents double-counting when two pol_map
# names map to the same pop2022 LGA within the same constituency
# e.g. Yamaltu + Deba → "Yamaltu/Deba" in the same constituency
pol_map_final = pol_map_final.drop_duplicates(
    subset=['FED_CONST', 'LGA_MERGE']).reset_index(drop=True)

# Count how many constituencies share each LGA -> used to allocate population
# Handles all splits uniformly:
#   Normal LGA in 1 constituency   -> N=1 -> full population 
#   Mushin I / Mushin II           -> N=2 -> half each
#   Oshodi-Isolo across I and II   -> N=2 -> half each
#   Kolokuma/Opokuma in 1 const.   -> N=1 (after dedup above)
lga_count = (pol_map_final
             .groupby(['STATE', 'LGA_MERGE'])['FED_CONST']
             .count()
             .reset_index(name='N_CONSTITUENCIES'))
pol_map_final = pol_map_final.merge(lga_count, on=['STATE', 'LGA_MERGE'], how='left')

# Add Taraba Disputed Areas with a placeholder constituency
disputed = pd.DataFrame([{
    'STATE':          'TARABA',
    'LGA':            'Disputed Areas',
    'SEN_DIST':       'N/A',
    'FED_CONST':      'N/A',
    'IS_SPLIT':       False,
    'LGA_MERGE':      'DISPUTED AREAS',
    'N_CONSTITUENCIES': 1
}])

pol_map_final = pd.concat([pol_map_final, disputed], ignore_index=True)

print(f"Unique constituencies: {pol_map_final['FED_CONST'].nunique()}  (should be 360)")
print(f"Unique states:         {pol_map_final['STATE'].nunique()}  (should be 37)")

Unique constituencies: 361  (should be 360)
Unique states:         37  (should be 37)


## 5. Constituency Population Estimates

In [203]:
# KEY FIX: merge on LGA_UPPER (uppercase) from merged_final
# merged_final['LGA'] is mixed case; pol_map_final['LGA_MERGE'] is uppercase.
# We pre-created LGA_UPPER = LGA.str.upper() in merged_final for exactly this merge.
pol_merged = pol_map_final.merge(
    merged_final[['ADM1_NAME', 'LGA_UPPER', 'POP25_PROJECTED']],
    left_on=['STATE', 'LGA_MERGE'],
    right_on=['ADM1_NAME', 'LGA_UPPER'],
    how='left'
)

# Diagnostic: check for unmatched LGAs
unmatched = pol_merged[pol_merged['POP25_PROJECTED'].isna()]
print(f"Unmatched LGAs: {len(unmatched)}  (should be 0)")
if len(unmatched) > 0:
    print(unmatched[['STATE', 'LGA_MERGE', 'FED_CONST']].to_string())

Unmatched LGAs: 0  (should be 0)


In [204]:
# Population share = LGA population ÷ number of constituencies sharing it
pol_merged['POP25_CONST'] = pol_merged['POP25_PROJECTED'] / pol_merged['N_CONSTITUENCIES']

# Aggregate to federal constituency level
const_pop = pol_merged.groupby(['STATE', 'FED_CONST'], as_index=False)['POP25_CONST'].sum()

# Sanity checks
lga_total   = merged_final['POP25_PROJECTED'].sum()
const_total = const_pop['POP25_CONST'].sum()
print(f"LGA total POP25:         {lga_total:,.0f}")
print(f"Constituency total POP25:{const_total:,.0f}")
print(f"Difference:              {abs(lga_total - const_total):,.0f}  (should be ~0)")
print(f"\nConstituencies with data: {const_pop['POP25_CONST'].notna().sum()} of {len(const_pop)}")
print(const_pop.head(20))

LGA total POP25:         235,207,341
Constituency total POP25:235,207,341
Difference:              0  (should be ~0)

Constituencies with data: 361 of 361
        STATE                            FED_CONST    POP25_CONST
0        ABIA                  Aba North/Aba South  834148.048287
1        ABIA                     Arochukwu/Ohafia  648448.251419
2        ABIA                                Bende  300743.608669
3        ABIA  Isiala Ngwa North/Isiala Ngwa South  453956.662782
4        ABIA                Isuikwuato/Umunneochi  435408.126490
5        ABIA           Obingwa/Ugwunagbo/Osisioma  761776.591655
6        ABIA                  Ukwa East/Ukwa West  227192.765337
7        ABIA  Umuahia North/Umuahia South/Ikwuano  780753.995839
8     ADAMAWA                  Demsa/Numan/Lamurde  637003.249603
9     ADAMAWA                          Fufore/Song  676118.434687
10    ADAMAWA                       Guyuk/Shelleng  543061.794015
11    ADAMAWA                           Hong/Gombi  5

## 6. Save Outputs

In [205]:
with pd.ExcelWriter(OUTPUT_DIR / 'nga_pop_estimates.xlsx', engine='openpyxl') as writer:
    merged_final.drop(columns=['LGA_UPPER']).to_excel(
        writer, index=False, sheet_name='popest_lga')
    const_pop.to_excel(
        writer, index=False, sheet_name='popest_const')

print("Saved nga_pop_estimates.xlsx with sheets: popest_lga, popest_const")

Saved nga_pop_estimates.xlsx with sheets: popest_lga, popest_const
